# Getting started — Speech-to-Text Model

Whisper `base.en` on SageMaker. Send WAV, MP3, or JSON (base64) to `/invocations` and get `{"text": "..."}`.

This notebook deploys the Marketplace model package to a real-time endpoint, transcribes a short clip, then deletes the endpoint. Real-time endpoints are on-demand (not spot) and bill until deleted.

**Before you run this:**
1. Subscribe on AWS Marketplace (**Continue to Subscribe** → **Continue to Configuration**).
2. Paste the **Model Package ARN** from that page into the next cell. That ARN is AWS's copy of the package (unique per region), not the seller ARN.
3. Run from SageMaker Studio / a notebook instance (execution role auto-detected), or set `ROLE_ARN` yourself. The role needs `sagemaker:CreateModel`, `sagemaker:CreateEndpoint`, and `sagemaker-runtime:InvokeEndpoint`.

Product page: https://www.sigmodata.com/products/speech-to-text-model  
Download this notebook: https://www.sigmodata.com/products/speech-to-text-model/getting-started.ipynb

In [ ]:
import json
import time
from pathlib import Path

import boto3
from botocore.config import Config

# Product ARN from AWS Marketplace → Continue to Configuration.
# That is AWS's copy of the model package (unique per region). Do not use a
# seller-account package ARN.
MODEL_PACKAGE_ARN = ""
assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), (
    "Paste the Model Package ARN from Marketplace Continue to Configuration"
)
REGION = MODEL_PACKAGE_ARN.split(":")[3]
INSTANCE_TYPE = "ml.g5.xlarge"  # or ml.m5.xlarge for CPU; ml.g4dn.xlarge if quota allows
AUDIO = Path("clip.wav")  # WAV or MP3, under 6 MiB for real-time

session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
runtime = session.client(
    "sagemaker-runtime",
    config=Config(read_timeout=600, retries={"max_attempts": 0}),
)
sts = session.client("sts")
account = sts.get_caller_identity()["Account"]
ROLE_ARN = (
    f"arn:aws:iam::{account}:role/service-role/AmazonSageMaker-ExecutionRole"
)
print("account", account)
print("role   ", ROLE_ARN)
print("arn    ", MODEL_PACKAGE_ARN)

## 1. Deploy a real-time endpoint

SageMaker starts the container with a literal `serve` argument: only `GET /ping`, `POST /invocations`, and `GET /execution-parameters`. No UI, no `/inference`, no auth.

Startup can take several minutes while the model loads. `ContainerStartupHealthCheckTimeoutInSeconds=600` covers that.

In [ ]:
name = f"sigmodata-stt-{int(time.time())}"
print("endpoint", name)

sm.create_model(
    ModelName=name,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InitialInstanceCount": 1,
        "InstanceType": INSTANCE_TYPE,
        "ContainerStartupHealthCheckTimeoutInSeconds": 600,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
print("Waiting for InService...")
sm.get_waiter("endpoint_in_service").wait(
    EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 60}
)
print(sm.describe_endpoint(EndpointName=name)["EndpointStatus"])

## 2. Transcribe

Raw bytes (`audio/wav` or `audio/mpeg`) or JSON `{"audio": "<base64>", "filename": "clip.wav"}`. Response is JSON with a `text` field. Real-time payload cap is **6 MiB**; use batch transform for longer files.

In [ ]:
def invoke(body: bytes, content_type: str) -> dict:
    resp = runtime.invoke_endpoint(
        EndpointName=name,
        ContentType=content_type,
        Accept="application/json",
        Body=body,
    )
    return json.loads(resp["Body"].read())

raw = AUDIO.read_bytes()
ct = "audio/mpeg" if AUDIO.suffix.lower() == ".mp3" else "audio/wav"
print(f"raw {AUDIO.name} ({ct}, {len(raw)} bytes)")
result = invoke(raw, ct)
print(result.get("text") or json.dumps(result, indent=2))

In [ ]:
import base64

payload = json.dumps({
    "audio": base64.b64encode(raw).decode(),
    "filename": AUDIO.name,
}).encode()
if len(payload) > 6 * 1024 * 1024:
    print(f"skip JSON: {len(payload)} bytes > 6 MiB")
else:
    result = invoke(payload, "application/json")
    print(result.get("text") or json.dumps(result, indent=2))

## 3. Clean up the endpoint

Real-time endpoints keep billing until deleted. Always run this when finished.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.get_waiter("endpoint_deleted").wait(
    EndpointName=name, WaiterConfig={"Delay": 15, "MaxAttempts": 40}
)
sm.delete_endpoint_config(EndpointConfigName=name)
sm.delete_model(ModelName=name)
print("deleted", name)

## Batch transform (optional)

Cheaper for bulk audio: put objects in S3 and run a transform job. This model is `SINGLE_RECORD` / `MaxPayloadInMB=6` (see `GET /execution-parameters`). One audio file per record; `ContentType` `audio/wav` or `audio/mpeg`.

In [ ]:
# Upload AUDIO to S3, then run a SingleRecord transform (6 MiB max per object).
bucket = "<your-bucket>"
prefix = "stt-batch-example"
s3 = session.client("s3")
s3.upload_file(str(AUDIO), bucket, f"{prefix}/{AUDIO.name}")
input_s3 = f"s3://{bucket}/{prefix}/"
output_s3 = f"s3://{bucket}/{prefix}/out/"
job = f"stt-batch-{int(time.time())}"
print("input ", input_s3)
print("output", output_s3)
# sm.create_transform_job(
#     TransformJobName=job,
#     ModelName=name,  # keep the model; skip endpoint if you only batch
#     MaxConcurrentTransforms=1,
#     MaxPayloadInMB=6,
#     BatchStrategy="SingleRecord",
#     TransformInput={
#         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": input_s3}},
#         "ContentType": "audio/wav",
#         "SplitType": "None",
#     },
#     TransformOutput={"S3OutputPath": output_s3, "Accept": "application/json"},
#     TransformResources={"InstanceType": "ml.m5.xlarge", "InstanceCount": 1},
# )

## Next steps

- Product page: https://www.sigmodata.com/products/speech-to-text-model
- Self-hosted UI, extra Whisper sizes, filesystem batch: [Speech-to-Text Server](https://www.sigmodata.com/products/speech-to-text-server)
- Support: support@sigmodata.com